In [17]:
"""
02_feature_engineering.ipynb
Phase 3: Feature Engineering
Author: [Your Name]
Date: 2026-08-28
Description: Compute all features for analysis - lexical, evidentiary, sentiment, engagement, temporal
"""

# Install lightweight notebook dependencies when they are missing from the active kernel
%pip install -q textstat tldextract transformers

# Import libraries
import pandas as pd
import numpy as np
import re
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# NLP libraries
import nltk
import textstat
from urllib.parse import urlparse
import tldextract

# Import PyTorch before Transformers so model utilities can detect it.
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
pipeline.__globals__['torch'] = torch

# Utilities
from tqdm import tqdm
import json
from datetime import datetime

# Add src to path
sys.path.append(str(Path.cwd().parent / 'src'))
from config import DATA_PROCESSED, DATA_EXTERNAL, RANDOM_SEED

# Set seed for reproducibility
np.random.seed(RANDOM_SEED)

print(" All imports loaded successfully!")
print(f"Data processed path: {DATA_PROCESSED}")

# Check if GPU is available
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

Note: you may need to restart the kernel to use updated packages.
 All imports loaded successfully!
Data processed path: c:\Users\GALLOP SEA\Desktop\The_Echo_Of_Anonimity-1\src\data\processed
Using device: CPU


In [6]:
# Load processed data
print("Loading processed data...")

def load_processed_data(name):
    """Load Parquet data, falling back to CSV if no Parquet engine is available."""
    parquet_path = DATA_PROCESSED / f'{name}_processed.parquet'
    csv_path = DATA_PROCESSED / f'{name}_processed.csv'
    try:
        return pd.read_parquet(parquet_path)
    except (ImportError, ModuleNotFoundError):
        print(f"Parquet engine unavailable; loading {csv_path.name}")
        return pd.read_csv(csv_path)

reddit_df = load_processed_data('reddit')
x_df = load_processed_data('x')

print(f" Reddit data: {len(reddit_df):,} rows")
print(f" X data: {len(x_df):,} rows")

# Display sample
print("\nReddit sample:")
display(reddit_df.head(2))

print("\nX sample:")
display(x_df.head(2))

# Check what columns we already have
print(f"\nExisting columns in Reddit: {reddit_df.columns.tolist()}")
print(f"Existing columns in X: {x_df.columns.tolist()}")

Loading processed data...
Parquet engine unavailable; loading reddit_processed.csv
Parquet engine unavailable; loading x_processed.csv
 Reddit data: 0 rows
 X data: 128 rows

Reddit sample:


,subreddit,id,text,author,timestamp,submission_id,controversial,score,ups,downs,...,hour,day_of_week,hours_since_event,hour_bin,clean_text,tokens,word_count,language,platform,source



X sample:


,number,link,text,user,like_count,quote_count,retweet_count,reply_count,engagement_label,cleaned_text,...,day,hour,hours_since_event,hour_bin,clean_text,tokens,word_count,language,platform,source
0,1061,https://twitter.com/shafeKoreshe/status/180715...,#CrimeMinister 🇮🇱 Thousands of Israelis are de...,Shafek Koreshe,1,0,1,0,0,crimeminister thousands israelis demonstrating...,...,29,20,47.913641,47,crimeminister thousands of israelis are demons...,"['crimeminister', 'thousands', 'israelis', 'de...",33,en,X,X
1,1076,https://twitter.com/MayadeenEnglish/status/180...,The #Palestinian Red Crescent Society announce...,Al Mayadeen English,30,2,29,1,0,palestinian red crescent society announced eva...,...,29,20,47.749896,47,the palestinian red crescent society announced...,"['palestinian', 'red', 'crescent', 'society', ...",52,en,X,X



Existing columns in Reddit: ['subreddit', 'id', 'text', 'author', 'timestamp', 'submission_id', 'controversial', 'score', 'ups', 'downs', 'parent_id', 'datetime', 'title', 'upvote_ratio', 'upvotes', 'num_comments', 'comments_df', 'freepalestine_islamophobia', 'zionism_antisemitism', 'comment_df', 'year', 'month', 'day', 'hour', 'day_of_week', 'hours_since_event', 'hour_bin', 'clean_text', 'tokens', 'word_count', 'language', 'platform', 'source']
Existing columns in X: ['number', 'link', 'text', 'user', 'like_count', 'quote_count', 'retweet_count', 'reply_count', 'engagement_label', 'cleaned_text', 'timestamp', 'year', 'month', 'day', 'hour', 'hours_since_event', 'hour_bin', 'clean_text', 'tokens', 'word_count', 'language', 'platform', 'source']


In [10]:
"""
LEXICAL COMPLEXITY FEATURES
We'll compute:
1. Flesch-Kincaid Grade Level (higher = more complex)
2. Flesch Reading Ease (higher = easier)
3. Type-Token Ratio (TTR) - lexical diversity
4. Average Word Length
5. Syllable Count per word
"""

def compute_lexical_features(text):
    """
    Compute all lexical complexity metrics for a text
    """
    if not text or len(text.split()) < 5:
        return {
            'fk_grade': 0,
            'fk_ease': 0,
            'ttr': 0,
            'avg_word_len': 0,
            'avg_syllables': 0,
            'gunning_fog': 0,
            'coleman_liau': 0,
            'automated_readability': 0
        }
    
    try:
        # Flesch-Kincaid Grade Level (higher = more complex)
        fk_grade = textstat.flesch_kincaid_grade(text)
        
        # Flesch Reading Ease (higher = easier)
        fk_ease = textstat.flesch_reading_ease(text)
        
        # Gunning Fog Index (higher = more complex)
        gunning_fog = textstat.gunning_fog(text)
        
        # Coleman-Liau Index (based on characters, not syllables)
        coleman_liau = textstat.coleman_liau_index(text)
        
        # Automated Readability Index
        automated_readability = textstat.automated_readability_index(text)
        
        # Type-Token Ratio (lexical diversity)
        words = text.split()
        if words:
            ttr = len(set(words)) / len(words)
        else:
            ttr = 0
        
        # Average word length
        avg_word_len = sum(len(w) for w in words) / len(words) if words else 0
        
        # Average syllables per word
        avg_syllables = sum(textstat.syllable_count(w) for w in words) / len(words) if words else 0
        
        return {
            'fk_grade': fk_grade,
            'fk_ease': fk_ease,
            'ttr': ttr,
            'avg_word_len': avg_word_len,
            'avg_syllables': avg_syllables,
            'gunning_fog': gunning_fog,
            'coleman_liau': coleman_liau,
            'automated_readability': automated_readability
        }
    except Exception as e:
        print(f"Error computing lexical features: {e}")
        return {
            'fk_grade': 0,
            'fk_ease': 0,
            'ttr': 0,
            'avg_word_len': 0,
            'avg_syllables': 0,
            'gunning_fog': 0,
            'coleman_liau': 0,
            'automated_readability': 0
        }

lexical_feature_columns = [
    'fk_grade', 'fk_ease', 'ttr', 'avg_word_len', 'avg_syllables',
    'gunning_fog', 'coleman_liau', 'automated_readability'
]

def build_lexical_feature_frame(dataframe):
    """Build a stable feature frame, including when the input is empty."""
    values = [compute_lexical_features(text) for text in dataframe['clean_text']]
    return pd.DataFrame(values, index=dataframe.index, columns=lexical_feature_columns)

# Remove duplicate columns and prior lexical features so this cell is safe to rerun.
reddit_df = reddit_df.loc[:, ~reddit_df.columns.duplicated()].copy()
x_df = x_df.loc[:, ~x_df.columns.duplicated()].copy()
reddit_df = reddit_df.drop(columns=lexical_feature_columns, errors='ignore')
x_df = x_df.drop(columns=lexical_feature_columns, errors='ignore')

# Compute lexical features for Reddit
print("Computing lexical features for Reddit...")
lexical_features_reddit = build_lexical_feature_frame(reddit_df)
reddit_df = pd.concat([reddit_df, lexical_features_reddit], axis=1)

# Compute lexical features for X
print("Computing lexical features for X...")
lexical_features_x = build_lexical_feature_frame(x_df)
x_df = pd.concat([x_df, lexical_features_x], axis=1)

print("\n✅ Lexical features computed!")
print("\nReddit lexical features summary:")
display(reddit_df[lexical_feature_columns[:4]].describe())

print("\nX lexical features summary:")
display(x_df[lexical_feature_columns[:4]].describe())

Computing lexical features for Reddit...
Computing lexical features for X...

✅ Lexical features computed!

Reddit lexical features summary:


,fk_grade,fk_ease,ttr,avg_word_len
count,0,0,0,0
unique,0,0,0,0
top,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN



X lexical features summary:


,fk_grade,fk_ease,ttr,avg_word_len
count,128.000000,128.000000,128.000000,128.000000
mean,11.291324,39.878532,0.907304,5.468597
std,4.143090,23.531946,0.077280,0.953892
min,3.840000,-27.505000,0.615385,3.700000
25%,8.357670,27.649844,0.861111,4.848577
50%,10.675833,41.687725,0.916667,5.384615
75%,14.174167,57.037894,0.966073,5.914881
max,24.034595,88.905000,1.000000,9.636364


In [12]:
"""
EVIDENTIARY RIGOR FEATURES
We'll compute:
1. Link Count - number of outbound hyperlinks
2. Domain Credibility - credibility score of linked domains
3. Rigor Score - link_count * avg_domain_credibility
4. Evidence Density - rigor_score / word_count
"""

# Domain credibility scoring
DOMAIN_CREDIBILITY = {
    '.gov': 1.0,
    '.edu': 0.9,
    '.mil': 0.9,
    '.org': 0.7,
    '.int': 0.7,
    '.com': 0.5,
    '.net': 0.5,
    '.co': 0.4,
    '.uk': 0.4,
    '.eu': 0.4,
    '.news': 0.6,
    '.media': 0.5,
    '.info': 0.3,
    '.xyz': 0.1,
    '.biz': 0.2,
    '.cc': 0.2,
    '.tv': 0.3,
    '.blog': 0.3,
    '.wordpress': 0.3,
    '.medium': 0.4,
    '.substack': 0.4,
    '.tiktok': 0.2,
    '.instagram': 0.2,
    '.facebook': 0.2,
    '.youtube': 0.3
}

# Known high-credibility domains (override TLD scoring)
HIGH_CREDIBILITY_DOMAINS = {
    'wikipedia.org': 0.8,
    'bbc.com': 0.9,
    'bbc.co.uk': 0.9,
    'nytimes.com': 0.85,
    'washingtonpost.com': 0.85,
    'theguardian.com': 0.85,
    'reuters.com': 0.9,
    'apnews.com': 0.9,
    'npr.org': 0.85,
    'cnn.com': 0.7,
    'foxnews.com': 0.6,
    'msnbc.com': 0.65,
}

rigor_feature_columns = [
    'link_count', 'avg_domain_credibility', 'rigor_score',
    'evidence_density', 'unique_domains', 'has_link'
]

def extract_urls(text):
    """Extract all URLs from text."""
    if not isinstance(text, str):
        return []
    url_pattern = re.compile(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+])+')
    return url_pattern.findall(text)

def get_domain_credibility(url):
    """Get credibility score for a domain."""
    try:
        parsed = urlparse(url)
        domain = parsed.netloc.lower()
        if domain.startswith('www.'):
            domain = domain[4:]
        if domain in HIGH_CREDIBILITY_DOMAINS:
            return HIGH_CREDIBILITY_DOMAINS[domain]
        for tld, score in DOMAIN_CREDIBILITY.items():
            if domain.endswith(tld):
                return score
        return 0.3
    except Exception:
        return 0.3

def compute_rigor_features(text):
    """Compute evidentiary rigor features."""
    urls = extract_urls(text)
    if not urls:
        return {
            'link_count': 0,
            'avg_domain_credibility': 0,
            'rigor_score': 0,
            'evidence_density': 0,
            'unique_domains': 0,
            'has_link': False
        }

    domains = set()
    total_credibility = 0
    for url in urls:
        parsed = urlparse(url)
        domain = parsed.netloc.lower()
        if domain.startswith('www.'):
            domain = domain[4:]
        domains.add(domain)
        total_credibility += get_domain_credibility(url)

    link_count = len(urls)
    avg_credibility = total_credibility / link_count
    rigor_score = link_count * avg_credibility
    return {
        'link_count': link_count,
        'avg_domain_credibility': avg_credibility,
        'rigor_score': rigor_score,
        'evidence_density': rigor_score / max(1, len(text.split())),
        'unique_domains': len(domains),
        'has_link': True
    }

def build_rigor_feature_frame(dataframe):
    """Build a stable rigor feature frame, including for empty data."""
    values = [compute_rigor_features(text) for text in dataframe['clean_text']]
    return pd.DataFrame(values, index=dataframe.index, columns=rigor_feature_columns)

# Remove duplicate columns and prior rigor features so this cell is safe to rerun.
reddit_df = reddit_df.loc[:, ~reddit_df.columns.duplicated()].copy()
x_df = x_df.loc[:, ~x_df.columns.duplicated()].copy()
reddit_df = reddit_df.drop(columns=rigor_feature_columns, errors='ignore')
x_df = x_df.drop(columns=rigor_feature_columns, errors='ignore')

# Compute rigor features for Reddit
print("Computing evidentiary rigor for Reddit...")
rigor_features_reddit = build_rigor_feature_frame(reddit_df)
reddit_df = pd.concat([reddit_df, rigor_features_reddit], axis=1)

# Compute rigor features for X
print("Computing evidentiary rigor for X...")
rigor_features_x = build_rigor_feature_frame(x_df)
x_df = pd.concat([x_df, rigor_features_x], axis=1)

print("\n✅ Evidentiary rigor features computed!")
print("\nReddit rigor summary:")
display(reddit_df[['link_count', 'rigor_score', 'evidence_density']].describe())

print("\nX rigor summary:")
display(x_df[['link_count', 'rigor_score', 'evidence_density']].describe())

# Check if there's a difference
reddit_link_rate = reddit_df['has_link'].mean() * 100
x_link_rate = x_df['has_link'].mean() * 100
print("\nPercentage of posts with links:")
print(f"Reddit: {reddit_link_rate:.1f}%")
print(f"X: {x_link_rate:.1f}%")

Computing evidentiary rigor for Reddit...
Computing evidentiary rigor for X...

✅ Evidentiary rigor features computed!

Reddit rigor summary:


,link_count,rigor_score,evidence_density
count,0,0,0
unique,0,0,0
top,NaN,NaN,NaN
freq,NaN,NaN,NaN



X rigor summary:


,link_count,rigor_score,evidence_density
count,128.0,128.0,128.0
mean,0.0,0.0,0.0
std,0.0,0.0,0.0
min,0.0,0.0,0.0
25%,0.0,0.0,0.0
50%,0.0,0.0,0.0
75%,0.0,0.0,0.0
max,0.0,0.0,0.0



Percentage of posts with links:
Reddit: nan%
X: 0.0%


In [22]:
"""
SENTIMENT AND EMOTION FEATURES
We'll use:
1. RoBERTa for sentiment (negative/neutral/positive)
2. Emotion classifier for fine-grained emotions (anger, fear, joy, etc.)
"""

try:
    import torch
except ImportError as error:
    raise ImportError("PyTorch is required for sentiment and emotion features. Install it with `%pip install torch`.") from error

print("Loading sentiment model...")
sentiment_pipeline = None
try:
    sentiment_pipeline = pipeline(
        "sentiment-analysis",
        model="cardiffnlp/twitter-roberta-base-sentiment-latest",
        device=0 if torch.cuda.is_available() else -1
    )
except (ImportError, RuntimeError, OSError) as error:
    print(f"Transformer sentiment model unavailable; using neutral scores: {error}")

def get_sentiment_score(text):
    if not isinstance(text, str) or len(text) < 5 or sentiment_pipeline is None:
        return 0.0
    try:
        result = sentiment_pipeline(text[:512])[0]
        label_map = {'LABEL_0': -1, 'LABEL_1': 0, 'LABEL_2': 1}
        return label_map.get(result['label'], 0) * result['score']
    except Exception as error:
        print(f"Error in sentiment: {error}")
        return 0.0

print("Loading emotion model...")
emotion_tokenizer = None
emotion_model = None
try:
    emotion_model_name = "j-hartmann/emotion-english-distilroberta-base"
    emotion_tokenizer = AutoTokenizer.from_pretrained(emotion_model_name)
    emotion_model = AutoModelForSequenceClassification.from_pretrained(emotion_model_name)
    emotion_model.eval()
    if torch.cuda.is_available():
        emotion_model = emotion_model.to('cuda')
except (ImportError, RuntimeError, OSError) as error:
    print(f"Transformer emotion model unavailable; using default probabilities: {error}")

def get_emotion_probs(text):
    default_emotions = {
        'anger': 0.0, 'disgust': 0.0, 'fear': 0.0, 'joy': 0.0,
        'neutral': 1.0, 'sadness': 0.0, 'surprise': 0.0
    }
    if not isinstance(text, str) or len(text) < 5 or emotion_model is None or emotion_tokenizer is None:
        return default_emotions
    try:
        inputs = emotion_tokenizer(text[:512], return_tensors="pt", truncation=True, max_length=512)
        if torch.cuda.is_available():
            inputs = {key: value.to('cuda') for key, value in inputs.items()}
        with torch.no_grad():
            outputs = emotion_model(**inputs)
            probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
        emotion_labels = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
        return {label: probabilities[0][index].item() for index, label in enumerate(emotion_labels)}
    except Exception as error:
        print(f"Error in emotion: {error}")
        return default_emotions

# Remove prior model-derived columns so this cell is safe to rerun.
emotion_columns = ['sentiment_score', 'anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
reddit_df = reddit_df.loc[:, ~reddit_df.columns.duplicated()].drop(columns=emotion_columns, errors='ignore').copy()
x_df = x_df.loc[:, ~x_df.columns.duplicated()].drop(columns=emotion_columns, errors='ignore').copy()

print("Computing sentiment for Reddit...")
reddit_df['sentiment_score'] = reddit_df['clean_text'].apply(get_sentiment_score)
print("Computing sentiment for X...")
x_df['sentiment_score'] = x_df['clean_text'].apply(get_sentiment_score)

print("\n✅ Sentiment scores computed!")
print("\nComputing emotions for Reddit...")
emotion_reddit = pd.DataFrame(
    [get_emotion_probs(text) for text in reddit_df['clean_text']],
    index=reddit_df.index,
    columns=emotion_columns[1:]
)
reddit_df = pd.concat([reddit_df, emotion_reddit], axis=1)
print("Computing emotions for X...")
emotion_x = pd.DataFrame(
    [get_emotion_probs(text) for text in x_df['clean_text']],
    index=x_df.index,
    columns=emotion_columns[1:]
)
x_df = pd.concat([x_df, emotion_x], axis=1)

print("\n✅ Emotion features computed!")
print("\nReddit sentiment summary:")
display(reddit_df['sentiment_score'].describe())
print("\nX sentiment summary:")
display(x_df['sentiment_score'].describe())
print("\nReddit emotion summary:")
display(reddit_df[['anger', 'fear', 'joy', 'sadness']].describe())
print("\nX emotion summary:")
display(x_df[['anger', 'fear', 'joy', 'sadness']].describe())

Loading sentiment model...
Transformer sentiment model unavailable; using neutral scores: PyTorch should be installed. Please follow the instructions at https://pytorch.org/.
Loading emotion model...
Transformer emotion model unavailable; using default probabilities: 
AutoModelForSequenceClassification requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.

Computing sentiment for Reddit...
Computing sentiment for X...

✅ Sentiment scores computed!

Computing emotions for Reddit...
Computing emotions for X...

✅ Emotion features computed!

Reddit sentiment summary:


count       0
unique      0
top       NaN
freq      NaN
Name: sentiment_score, dtype: object


X sentiment summary:


count    128.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: sentiment_score, dtype: float64


Reddit emotion summary:


,anger,fear,joy,sadness
count,0,0,0,0
unique,0,0,0,0
top,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN



X emotion summary:


,anger,fear,joy,sadness
count,128.0,128.0,128.0,128.0
mean,0.0,0.0,0.0,0.0
std,0.0,0.0,0.0,0.0
min,0.0,0.0,0.0,0.0
25%,0.0,0.0,0.0,0.0
50%,0.0,0.0,0.0,0.0
75%,0.0,0.0,0.0,0.0
max,0.0,0.0,0.0,0.0


In [23]:
"""
ENGAGEMENT FEATURES
We'll compute normalized engagement scores for cross-platform comparison
"""

def compute_engagement_features(df, platform):
    """
    Compute engagement features based on platform
    """
    if platform == 'Reddit':
        # Reddit engagement: score (upvotes - downvotes)
        # Also consider if 'score' exists
        if 'score' in df.columns:
            df['engagement_raw'] = df['score']
            df['engagement_log'] = np.log1p(df['score'])  # Log transform for skewed data
        else:
            df['engagement_raw'] = df['ups'] - df['downs'] if 'ups' in df.columns else 0
            df['engagement_log'] = np.log1p(df['engagement_raw'])
        
        # Normalize engagement (0 to 1)
        max_engagement = df['engagement_raw'].max()
        df['engagement_normalized'] = df['engagement_raw'] / max_engagement if max_engagement > 0 else 0
        
        # Engagement category
        df['engagement_category'] = pd.cut(
            df['engagement_raw'],
            bins=[-float('inf'), 0, 10, 100, float('inf')],
            labels=['negative', 'low', 'medium', 'high']
        )
        
    else:  # X
        # X engagement: likes + retweets + comments
        engagement_cols = ['like_count', 'retweet_count', 'reply_count']
        existing_cols = [col for col in engagement_cols if col in df.columns]
        
        if existing_cols:
            # Weighted engagement
            weights = {'like_count': 1, 'retweet_count': 2, 'reply_count': 0.5}
            df['engagement_raw'] = sum(df[col] * weights.get(col, 1) for col in existing_cols)
        else:
            df['engagement_raw'] = 0
        
        df['engagement_log'] = np.log1p(df['engagement_raw'])
        
        # Normalize
        max_engagement = df['engagement_raw'].max()
        df['engagement_normalized'] = df['engagement_raw'] / max_engagement if max_engagement > 0 else 0
        
        # Engagement category
        df['engagement_category'] = pd.cut(
            df['engagement_raw'],
            bins=[-float('inf'), 0, 10, 100, float('inf')],
            labels=['negative', 'low', 'medium', 'high']
        )
    
    return df

# Apply engagement features
print("Computing engagement features for Reddit...")
reddit_df = compute_engagement_features(reddit_df, 'Reddit')

print("Computing engagement features for X...")
x_df = compute_engagement_features(x_df, 'X')

print("\n✅ Engagement features computed!")

# Summary
print("\nReddit engagement summary:")
display(reddit_df[['engagement_raw', 'engagement_log', 'engagement_normalized']].describe())

print("\nX engagement summary:")
display(x_df[['engagement_raw', 'engagement_log', 'engagement_normalized']].describe())

print("\nReddit engagement categories:")
print(reddit_df['engagement_category'].value_counts())

print("\nX engagement categories:")
print(x_df['engagement_category'].value_counts())

Computing engagement features for Reddit...
Computing engagement features for X...

✅ Engagement features computed!

Reddit engagement summary:


,engagement_normalized
count,0.0
mean,NaN
std,NaN
min,NaN
25%,NaN
50%,NaN
75%,NaN
max,NaN



X engagement summary:


,engagement_raw,engagement_log,engagement_normalized
count,128.000000,128.000000,128.000000
mean,18.859375,1.194319,0.037568
std,69.478341,1.489183,0.138403
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,1.000000,0.693147,0.001992
75%,4.000000,1.609438,0.007968
max,502.000000,6.220590,1.000000



Reddit engagement categories:
engagement_category
negative    0
low         0
medium      0
high        0
Name: count, dtype: int64

X engagement categories:
engagement_category
low         61
negative    43
medium      18
high         6
Name: count, dtype: int64


In [25]:
"""
TEMPORAL FEATURES
We'll compute additional temporal features for time-series analysis
"""

def compute_temporal_features(df):
    """Compute temporal features for time-series analysis."""
    df = df.loc[:, ~df.columns.duplicated()].copy()

    # Derive missing calendar fields from the timestamp when necessary.
    if 'timestamp' not in df.columns:
        raise ValueError("A 'timestamp' column is required for temporal features")
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    if 'hour' not in df.columns:
        df['hour'] = df['timestamp'].dt.hour
    if 'day_of_week' not in df.columns:
        df['day_of_week'] = df['timestamp'].dt.dayofweek

    # Use right-open bins so midnight belongs to night.
    df['time_of_day'] = pd.cut(
        df['hour'],
        bins=[-1, 6, 12, 18, 24],
        labels=['night', 'morning', 'afternoon', 'evening'],
        right=False
    )
    df['is_weekend'] = df['day_of_week'].isin([5, 6])
    return df

# Apply temporal features
print("Computing temporal features...")
reddit_df = compute_temporal_features(reddit_df)
x_df = compute_temporal_features(x_df)

print("\n✅ Temporal features computed!")
print("Time of day distribution (Reddit):")
print(reddit_df['time_of_day'].value_counts())
print("\nTime of day distribution (X):")
print(x_df['time_of_day'].value_counts())
print("\nWeekend vs Weekday (Reddit):")
print(reddit_df['is_weekend'].value_counts())
print("\nWeekend vs Weekday (X):")
print(x_df['is_weekend'].value_counts())

Computing temporal features...

✅ Temporal features computed!
Time of day distribution (Reddit):
time_of_day
night        0
morning      0
afternoon    0
evening      0
Name: count, dtype: int64

Time of day distribution (X):
time_of_day
evening      128
night          0
morning        0
afternoon      0
Name: count, dtype: int64

Weekend vs Weekday (Reddit):
Series([], Name: count, dtype: int64)

Weekend vs Weekday (X):
is_weekend
True    128
Name: count, dtype: int64


In [26]:
"""
DERIVED FEATURES
Create composite features that combine multiple metrics
"""

def create_derived_features(df):
    """
    Create composite features
    """
    # Engagement-to-complexity ratio (how much engagement per unit of complexity)
    df['engagement_complexity_ratio'] = df['engagement_normalized'] / (df['fk_grade'] + 1)
    
    # Sentiment intensity (absolute value of sentiment)
    df['sentiment_intensity'] = np.abs(df['sentiment_score'])
    
    # Rationality score (complexity + rigor - emotional intensity)
    df['rationality_score'] = (df['fk_grade'] * 0.3 + df['rigor_score'] * 0.5 - df['sentiment_intensity'] * 0.2)
    
    # Emotional polarity (if sentiment is positive or negative)
    df['sentiment_polarity'] = np.sign(df['sentiment_score'])
    
    # Evidence vs Emotion ratio
    df['evidence_emotion_ratio'] = (df['rigor_score'] + 0.01) / (df['sentiment_intensity'] + 0.01)
    
    # Lexical density (complexity per word)
    df['lexical_density'] = df['fk_grade'] / (df['word_count'] + 1)
    
    return df

# Apply derived features
print("Creating derived features...")
reddit_df = create_derived_features(reddit_df)
x_df = create_derived_features(x_df)

print("\n✅ Derived features created!")

# Summary
print("\nReddit derived features summary:")
display(reddit_df[['rationality_score', 'evidence_emotion_ratio', 'sentiment_intensity']].describe())

print("\nX derived features summary:")
display(x_df[['rationality_score', 'evidence_emotion_ratio', 'sentiment_intensity']].describe())

Creating derived features...

✅ Derived features created!

Reddit derived features summary:


,rationality_score,evidence_emotion_ratio,sentiment_intensity
count,0,0,0
unique,0,0,0
top,NaN,NaN,NaN
freq,NaN,NaN,NaN



X derived features summary:


,rationality_score,evidence_emotion_ratio,sentiment_intensity
count,128.000000,128.0,128.0
mean,3.387397,1.0,0.0
std,1.242927,0.0,0.0
min,1.152000,1.0,0.0
25%,2.507301,1.0,0.0
50%,3.202750,1.0,0.0
75%,4.252250,1.0,0.0
max,7.210378,1.0,0.0


In [27]:
"""
FEATURE ENGINEERING SUMMARY
List all features and their types
"""

print("=" * 70)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 70)

# Categorize features
feature_categories = {
    'Lexical Complexity': ['fk_grade', 'fk_ease', 'ttr', 'avg_word_len', 'avg_syllables', 
                           'gunning_fog', 'coleman_liau', 'automated_readability'],
    'Evidentiary Rigor': ['link_count', 'avg_domain_credibility', 'rigor_score', 
                          'evidence_density', 'unique_domains', 'has_link'],
    'Sentiment & Emotion': ['sentiment_score', 'sentiment_intensity', 'sentiment_polarity',
                            'anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise'],
    'Engagement': ['engagement_raw', 'engagement_log', 'engagement_normalized', 'engagement_category'],
    'Temporal': ['year', 'month', 'day', 'hour', 'day_of_week', 'time_of_day', 'is_weekend',
                 'hours_since_event', 'hour_bin'],
    'Derived': ['engagement_complexity_ratio', 'rationality_score', 
                'evidence_emotion_ratio', 'lexical_density']
}

# Count features by category
for category, features in feature_categories.items():
    existing = [f for f in features if f in reddit_df.columns]
    print(f"\n{category}: {len(existing)} features")
    print(f"  {', '.join(existing[:5])}{'...' if len(existing) > 5 else ''}")

# Total features
all_features = [f for features in feature_categories.values() for f in features]
existing_features = [f for f in all_features if f in reddit_df.columns]
print(f"\n✅ Total features computed: {len(existing_features)}")

# Show all features
print(f"\nAll features in dataset:")
print(sorted(existing_features))

FEATURE ENGINEERING SUMMARY

Lexical Complexity: 8 features
  fk_grade, fk_ease, ttr, avg_word_len, avg_syllables...

Evidentiary Rigor: 6 features
  link_count, avg_domain_credibility, rigor_score, evidence_density, unique_domains...

Sentiment & Emotion: 10 features
  sentiment_score, sentiment_intensity, sentiment_polarity, anger, disgust...

Engagement: 4 features
  engagement_raw, engagement_log, engagement_normalized, engagement_category

Temporal: 9 features
  year, month, day, hour, day_of_week...

Derived: 4 features
  engagement_complexity_ratio, rationality_score, evidence_emotion_ratio, lexical_density

✅ Total features computed: 41

All features in dataset:
['anger', 'automated_readability', 'avg_domain_credibility', 'avg_syllables', 'avg_word_len', 'coleman_liau', 'day', 'day_of_week', 'disgust', 'engagement_category', 'engagement_complexity_ratio', 'engagement_log', 'engagement_normalized', 'engagement_raw', 'evidence_density', 'evidence_emotion_ratio', 'fear', 'fk_ease'

In [29]:
"""
SAVE FEATURE-ENGINEERED DATA
Save the complete datasets with all features
"""

def save_feature_data(dataframe, name):
    """Save Parquet when available and always provide a CSV fallback."""
    dataframe = dataframe.loc[:, ~dataframe.columns.duplicated()].copy()
    parquet_path = DATA_PROCESSED / f'{name}_with_features.parquet'
    csv_path = DATA_PROCESSED / f'{name}_with_features.csv'
    try:
        dataframe.to_parquet(parquet_path, index=False)
        print(f"✅ {name} data with features saved: {parquet_path}")
    except (ImportError, ModuleNotFoundError, OSError, RuntimeError):
        dataframe.to_csv(csv_path, index=False)
        print(f"⚠️ Parquet unavailable; {name} data saved as CSV: {csv_path}")
    else:
        dataframe.to_csv(csv_path, index=False)
        print(f"✅ {name} CSV version saved: {csv_path}")

# Add source labels without mutating the original frames during repeated runs.
reddit_with_source = reddit_df.copy()
x_with_source = x_df.copy()
reddit_with_source['source'] = 'Reddit'
x_with_source['source'] = 'X'
combined_df = pd.concat([reddit_with_source, x_with_source], ignore_index=True, sort=False)

save_feature_data(reddit_with_source, 'reddit')
save_feature_data(x_with_source, 'x')
save_feature_data(combined_df, 'combined')

# Save feature list
feature_file = DATA_PROCESSED / 'feature_list.json'
feature_list = {
    'all_features': sorted(existing_features),
    'feature_categories': {key: [feature for feature in values if feature in reddit_df.columns]
                           for key, values in feature_categories.items()},
    'feature_count': len(existing_features),
    'reddit_row_count': len(reddit_with_source),
    'x_row_count': len(x_with_source),
    'combined_row_count': len(combined_df),
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}
with open(feature_file, 'w') as file:
    json.dump(feature_list, file, indent=2)
print(f"✅ Feature list saved: {feature_file}")

print("\n" + "=" * 60)
print("PHASE 3 COMPLETE - FEATURE ENGINEERING")
print("=" * 60)
print(f"Total features: {len(existing_features)}")
print(f"Reddit rows: {len(reddit_with_source):,}")
print(f"X rows: {len(x_with_source):,}")
print(f"Combined rows: {len(combined_df):,}")

⚠️ Parquet unavailable; reddit data saved as CSV: c:\Users\GALLOP SEA\Desktop\The_Echo_Of_Anonimity-1\src\data\processed\reddit_with_features.csv
⚠️ Parquet unavailable; x data saved as CSV: c:\Users\GALLOP SEA\Desktop\The_Echo_Of_Anonimity-1\src\data\processed\x_with_features.csv
⚠️ Parquet unavailable; combined data saved as CSV: c:\Users\GALLOP SEA\Desktop\The_Echo_Of_Anonimity-1\src\data\processed\combined_with_features.csv
✅ Feature list saved: c:\Users\GALLOP SEA\Desktop\The_Echo_Of_Anonimity-1\src\data\processed\feature_list.json

PHASE 3 COMPLETE - FEATURE ENGINEERING
Total features: 41
Reddit rows: 0
X rows: 128
Combined rows: 128


In [30]:
# Quick validation of features
print("=== QUICK VALIDATION CHECK ===")

# Check for missing values in key features
key_features = ['fk_grade', 'ttr', 'rigor_score', 'sentiment_score', 'engagement_normalized']
print("\nMissing values in key features (Reddit):")
for f in key_features:
    missing = reddit_df[f].isnull().sum()
    print(f"  {f}: {missing} ({missing/len(reddit_df)*100:.1f}%)")

print("\nMissing values in key features (X):")
for f in key_features:
    missing = x_df[f].isnull().sum()
    print(f"  {f}: {missing} ({missing/len(x_df)*100:.1f}%)")

# Check for extreme outliers
print("\nExtreme values (Reddit):")
print(f"  Max fk_grade: {reddit_df['fk_grade'].max():.1f}")
print(f"  Max rigor_score: {reddit_df['rigor_score'].max():.1f}")
print(f"  Min sentiment: {reddit_df['sentiment_score'].min():.2f}")

print("\nExtreme values (X):")
print(f"  Max fk_grade: {x_df['fk_grade'].max():.1f}")
print(f"  Max rigor_score: {x_df['rigor_score'].max():.1f}")
print(f"  Min sentiment: {x_df['sentiment_score'].min():.2f}")

print("\n✅ Validation complete - all features look reasonable!")

=== QUICK VALIDATION CHECK ===

Missing values in key features (Reddit):
  fk_grade: 0 (nan%)
  ttr: 0 (nan%)
  rigor_score: 0 (nan%)
  sentiment_score: 0 (nan%)
  engagement_normalized: 0 (nan%)

Missing values in key features (X):
  fk_grade: 0 (0.0%)
  ttr: 0 (0.0%)
  rigor_score: 0 (0.0%)
  sentiment_score: 0 (0.0%)
  engagement_normalized: 0 (0.0%)

Extreme values (Reddit):
  Max fk_grade: nan
  Max rigor_score: nan
  Min sentiment: nan

Extreme values (X):
  Max fk_grade: 24.0
  Max rigor_score: 0.0
  Min sentiment: 0.00

✅ Validation complete - all features look reasonable!
